In [4]:
import wandb
import matplotlib.pyplot as plt
import numpy as np
import os

def setup_aaai_fullpage_style():
    """Set matplotlib rcParams for AAAI full-page width plots."""
    plt.rcParams.update({
        'font.size': 12,
        'axes.labelsize': 12,
        'legend.fontsize': 11,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'font.family': 'serif',
        'axes.grid': True,
        'axes.axisbelow': True,
        'grid.alpha': 0.3,
        'grid.linewidth': 0.8,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.spines.left': True,
        'axes.spines.bottom': True,
        'axes.linewidth': 1.0,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'lines.markersize': 5,
        'lines.linewidth': 1.3,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.titlepad': 4,
    })

def fetch_histogram_steps(run, param_name):
    """
    Fetch list of (step, hist) for a given parameter histogram from a W&B run.
    """
    history = run.history(samples=10000, keys=[param_name])
    hist_steps = []
    hist_vals = []
    for i, row in history.iterrows():
        val = row[param_name]
        if isinstance(val, dict) and val.get("_type") == "histogram":
            # must have bins and counts/values (not just _type)
            if ("bins" in val and ("counts" in val or "values" in val)) or "histogram" in val:
                hist_steps.append(row['_step'] if '_step' in row else i)
                hist_vals.append(val)
    return hist_steps, hist_vals

def plot_histogram(hist, outpath, xlim=(-15, 15), bar_color="#9467bd", norm_height=True):
    """
    Plot a single histogram with AAAI full-page style.
    If norm_height is True, normalize bars to max=1 for better visibility.
    """
    setup_aaai_fullpage_style()
    fig, ax = plt.subplots(figsize=(6.5, 2.7), dpi=300)  # full page width AAAI is about 6.5in
    # Try common possible keys for wandb histogram
    if "counts" in hist:
        bins = np.array(hist['bins'])
        counts = np.array(hist['counts'])
    elif "values" in hist:
        bins = np.array(hist['bins'])
        counts = np.array(hist['values'])
    elif "histogram" in hist:
        bins = np.array(hist['histogram']['bins'])
        counts = np.array(hist['histogram']['values'])
    else:
        raise ValueError(f"Unknown histogram format: {hist.keys()}")

    # Optionally normalize for better visibility (especially for initial weights)
    if norm_height and counts.max() > 0:
        counts = counts / counts.max()

    ax.bar(bins[:-1], counts, width=np.diff(bins), align='edge', alpha=0.8, color=bar_color, edgecolor="black", linewidth=0.2)
    ax.set_xlim(xlim)
    ax.set_xlabel("Weight")
    ax.set_ylabel("Normalized count" if norm_height else "Count")
    ax.grid(True, axis="y")
    ax.tick_params(axis='x', which='both', direction='out')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout(pad=0.2)
    fig.savefig(outpath, format="pdf", bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)

def main():
    # User: edit these as needed
    ENTITY = "sbuehrer-eth-z-rich"
    PROJECT = "RDDLGN"
    RUN_ID = "i1cjqgzd"
    OUTPUT_DIR = "weight_histograms_aaai_fullpage"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # All layers to plot: (layer label, parameter name, file prefix)
    layer_params = [
        ("N Layers", "parameters/n_layers.0.weights", "n_layers"),
        ("K Layers", "parameters/k_layers.0.weights", "k_layers"),
        ("L Layers", "parameters/l_layers.0.weights", "l_layers"),
        ("P Layers", "parameters/p_layers.0.weights", "p_layers"),
        # For M layers, there are several sublayers; plot each as separate file
        ("M Layers 0", "parameters/m_layers.0.weights", "m_layers_0"),
        ("M Layers 1", "parameters/m_layers.1.weights", "m_layers_1"),
        ("M Layers 2", "parameters/m_layers.2.weights", "m_layers_2"),
        ("M Layers 3", "parameters/m_layers.3.weights", "m_layers_3"),
        ("M Layers 4", "parameters/m_layers.4.weights", "m_layers_4"),
        ("M Layers 5", "parameters/m_layers.5.weights", "m_layers_5"),
        ("M Layers 6", "parameters/m_layers.6.weights", "m_layers_6"),
        # Add embedding if desired
        ("Embedding", "parameters/embedding.weight", "embedding"),
    ]

    api = wandb.Api()
    run = api.run(f"{ENTITY}/{PROJECT}/{RUN_ID}")

    for layer_name, param_name, file_prefix in layer_params:
        print(f"Processing {layer_name}: {param_name}")
        steps, hists = fetch_histogram_steps(run, param_name)
        if not hists:
            print(f"  No valid histogram for {param_name}")
            continue
        # Store start and final as separate plots, always
        out_pdf_start = os.path.join(OUTPUT_DIR, f"weight_histogram_{file_prefix}_start.pdf")
        out_pdf_final = os.path.join(OUTPUT_DIR, f"weight_histogram_{file_prefix}_final.pdf")

        # Use normalization for both for better bar visibility (esp. start)
        plot_histogram(hists[0], out_pdf_start, xlim=(-15, 15), norm_height=True)
        plot_histogram(hists[-1], out_pdf_final, xlim=(-15, 15), norm_height=True)
        print(f"  Saved: {out_pdf_start} and {out_pdf_final}")

if __name__ == "__main__":
    main()

Processing N Layers: parameters/n_layers.0.weights
  Saved: weight_histograms_aaai_fullpage/weight_histogram_n_layers_start.pdf and weight_histograms_aaai_fullpage/weight_histogram_n_layers_final.pdf
Processing K Layers: parameters/k_layers.0.weights
  Saved: weight_histograms_aaai_fullpage/weight_histogram_k_layers_start.pdf and weight_histograms_aaai_fullpage/weight_histogram_k_layers_final.pdf
Processing L Layers: parameters/l_layers.0.weights
  Saved: weight_histograms_aaai_fullpage/weight_histogram_l_layers_start.pdf and weight_histograms_aaai_fullpage/weight_histogram_l_layers_final.pdf
Processing P Layers: parameters/p_layers.0.weights
  Saved: weight_histograms_aaai_fullpage/weight_histogram_p_layers_start.pdf and weight_histograms_aaai_fullpage/weight_histogram_p_layers_final.pdf
Processing M Layers 0: parameters/m_layers.0.weights
  Saved: weight_histograms_aaai_fullpage/weight_histogram_m_layers_0_start.pdf and weight_histograms_aaai_fullpage/weight_histogram_m_layers_0_fina